## Modeling the agent
1. Feature selection: remove the columns I will not use (for example, the indexing, lat, longitude, and name of the station)
2. Train the models with resampling
3. .....


### Libraries and tools to ues




### Preparation of the venvironment (as on the other notebooks)

In [7]:
%load_ext autoreload
%autoreload 2

import sys, os
from pathlib import Path

# Make src/ importable: add the project root (parent of notebooks/) to sys.path
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Run from the project root so relative paths like "data/raw/..." resolve
os.chdir(project_root)
print(f"Working dir: {os.getcwd()}")

from src.model import *

import pandas as pd

# Start by importing the data
train_path = Path('data/processed/cleaned_data.csv')
df_train = pd.read_csv(train_path)


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Working dir: c:\Users\rcsrc\Documents\GitHub\predictive_modeling_and_mobility_forecasting


### Feature selection

In [ ]:
# Split the features
x_train, y_train = prepare_features(df_train)
#print(X_train)

In [8]:
featurce_performance = feature_selection_report(x_train,y_train)

print("features:", featurce_performance)
print("train data shape:", x_train.shape)


         station_number  hour  minute  dayofweek  month  is_weekend  hour_sin  \
0                 32000    17      30          1      9           0 -0.965926   
1                 32001    17      30          1      9           0 -0.965926   
2                 32002    17      30          1      9           0 -0.965926   
3                 32003    17      30          1      9           0 -0.965926   
4                 32004    17      30          1      9           0 -0.965926   
...                 ...   ...     ...        ...    ...         ...       ...   
2150245           32275     8       0          3      3           0  0.866025   
2150246           32277     8       0          3      3           0  0.866025   
2150247           32278     8       0          3      3           0  0.866025   
2150248           32280     8       0          3      3           0  0.866025   
2150249           32283     8       0          3      3           0  0.866025   

         hour_cos  is_holid

In [6]:
x_train

,station_number,hour,minute,dayofweek,month,is_weekend,hour_sin,hour_cos,is_holiday
0,32000,17,30,1,9,0,-0.965926,-0.258819,False
1,32001,17,30,1,9,0,-0.965926,-0.258819,False
2,32002,17,30,1,9,0,-0.965926,-0.258819,False
3,32003,17,30,1,9,0,-0.965926,-0.258819,False
4,32004,17,30,1,9,0,-0.965926,-0.258819,False
...,...,...,...,...,...,...,...,...,...
2150245,32275,8,0,3,3,0,0.866025,-0.500000,False
2150246,32277,8,0,3,3,0,0.866025,-0.500000,False
2150247,32278,8,0,3,3,0,0.866025,-0.500000,False
2150248,32280,8,0,3,3,0,0.866025,-0.500000,False


### Benchmarking different methods

In [ ]:
#I first need to create all the models that I will asses
model_grid = get_model_grid()

In [ ]:
#Then i train them with the default folds
benchmark = benchmark_models(x=x_train,y=y_train,models=model_grid,sample_size=10_000)

In [ ]:
#Finally we print to see the higher performing
from src.model import plot_benchmark

fig, ax = plot_benchmark(results)

#### Now we train a model

In [ ]:
# check the featureless
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np
import joblib

from sklearn.dummy import DummyRegressor
baseline = DummyRegressor(strategy="median")
baseline.fit(X_train, y_train)

y_pred = baseline.predict(X_test)

print(y_pred)

In [ ]:
# This is a basic DecisionTreeRegressor, check if everything is working propperly, look into and crete a benchmark to determine if it improves with respect to a featureless learner.

from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error,mean_squared_log_error
import numpy as np
import joblib

# Initialize — set max_depth to keep it interpretable and avoid overfitting
model = DecisionTreeRegressor(
    max_depth=20,        # decision tree depth
    min_samples_leaf=10, # minimum observations per leaf
    random_state=42,
)

# Train
model.fit(X_train, y_train)

print(model.score(X_test, y_test))
# Predict
y_pred = model.predict(X_test)

print(y_pred)

# Evaluate
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
rmsle = np.sqrt(mean_squared_log_error(y_test, y_pred))
print(f"MAE:   {mae:.3f}")
print(f"RMSE:  {rmse:.3f}")
print(f"RMSLE: {rmsle:.3f}")

In [ ]:
# Save the model
joblib.dump(model, 'models/decision_tree_v1.joblib')